# DhakaFlow P2 — Simulated Sensor-Graph Generation (Colab)

**No API keys needed. CPU runtime is enough — this is SUMO, not training.**

Generates the `DHAKA_SIMULATED_SENSOR_GRAPH` forecasting dataset on the
frozen Lab 02 R1 network: 250 virtual sensors (deterministic placement,
largest-SCC restricted), 5-minute bins, 24 simulated hours × seeds, plus the
DCRNN-recipe adjacency. Output goes to Drive. The smoke run already passed
locally (`artifacts/p2_sensor_graph_smoke*`); this is the full run.

Labels: every file is SIMULATED / `DHAKA_SIMULATED_SENSOR_GRAPH` — never
described as real measurements. Demand is `SYNTHETIC_STRUCTURED_DEMAND`
(pre-calibration; the calibrated twin replaces it at the twin gate).

In [ ]:
# 1 · Environment — SUMO via pip (binaries included), no apt needed.
%pip -q install eclipse-sumo sumolib numpy
import subprocess
sumo_bin = subprocess.run(['which','sumo'], capture_output=True, text=True).stdout.strip()
print('sumo:', sumo_bin or 'NOT FOUND')
print(subprocess.run(['sumo','--version'], capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
# 2 · Mount Drive + clone the repo at the current commit.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/dhakaflow/artifacts/p2_sensor_graph'

REPO_DIR = '/content/thesis'
import os, subprocess, sys
if not os.path.exists(REPO_DIR):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/auth889-ai/thesis.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(['git','log','-1','--oneline'], capture_output=True, text=True).stdout)

In [ ]:
# 3 · Smoke first (repo rule) — ~1 min. Must finish with insertion 1.00.
!python scripts/run_sensor_graph.py --mode smoke --out /content/p2_smoke

In [ ]:
# 4 · FULL generation: 24 h × 3 seeds (three simulated days), straight to Drive.
!python scripts/run_sensor_graph.py --mode full --seeds 42 43 44 --out {DRIVE_OUT}

In [ ]:
# 5 · Verify + record PASS evidence on Drive.
import json, numpy as np, hashlib, pathlib
out = pathlib.Path(DRIVE_OUT)
m = json.loads((out/'MANIFEST.json').read_text())
z = np.load(out/'sensor_graph_data.npz')
data, mask, adj = z['data'], z['mask'], z['adjacency']
checks = {
  'evidence_domain_is_simulated': m['evidence_domain'] == 'DHAKA_SIMULATED_SENSOR_GRAPH',
  'nodes_in_mandate_100_300': 100 <= data.shape[1] <= 300,
  'three_days_of_5min_bins': data.shape[0] == 3*24*12,
  'four_channels': data.shape[2] == 4,
  'adjacency_square': adj.shape == (data.shape[1], data.shape[1]),
  'peak_bins_well_covered': float(mask[(np.arange(data.shape[0])%288)//12==8].mean()) > 0.5,
  'speed_nan_only_where_masked': bool(np.isnan(data[...,0])[mask].sum() == 0),
}
print(json.dumps(checks, indent=1))
assert all(checks.values()), 'P2 acceptance checks FAILED — do not mark PASS'
sha = hashlib.sha256((out/'sensor_graph_data.npz').read_bytes()).hexdigest()
(out/'P2_PASS_EVIDENCE.json').write_text(json.dumps({
  'checks': checks, 'data_sha256': sha,
  'manifest_node_hash': m['node_list_sha256']}, indent=2))
print('P2 acceptance checks all green — evidence written to Drive:', out/'P2_PASS_EVIDENCE.json')

## Next (P3, forecasting gate)

This dataset feeds the BasicTS adapter: chronological split, horizons
5/15/30 min, models Persistence → HA → RF/XGBoost → LSTM/GRU → DCRNN/GWNet/
STID/STAEformer → Chronos-2. No key needed except HF token for Chronos-2
(READY_FOR_KEY).